# Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import convolve
import os

from tqdm import trange

import brighteyes_ism.simulation.PSF_sim as sim
import brighteyes_ism.analysis.Graph_lib as gra
import brighteyes_ism.analysis.APR_lib as apr

from skimage.filters import gaussian
import brighteyes_ism.simulation.Tubulin_sim as simTub

import scipy
from scipy.ndimage import fourier_shift, shift
from scipy.fft import fft, ifft, ifftshift

# Functions

In [ ]:
def model(t, C, tau, T):
    offset = t[len(t)//2]
    modello = C * ( np.heaviside(t-offset, 1) + 1/(np.exp(T/tau) - 1) ) * np.exp((-t+offset)/tau) 
    return modello

def corrected_ref_model(t, C, C_R, tau, tau_R, T):
    diff_tau = 1/tau_R - 1/tau
    result = model(t, C, tau, T)
    tempo = np.zeros_like(t)
    tempo[len(t)//2] = result[len(t)//2]
    result = C/C_R * (  tempo + ( diff_tau ) * result )
    return result

def ref_model(t, C, C_R, tau, tau_R, dT, ref_data, T):
    amplitude = C/C_R
    diff_tau = 1/tau_R - 1/tau
    shifted_ref_data = shift(ref_data, dT, order=1, mode='grid-wrap')
    model_data = model(t, C, tau, T)
    correction = diff_tau * conv_circ( model_data, shifted_ref_data )
    return amplitude * ( shifted_ref_data + correction )

def rectangular_IRF(t, dt):
    """
    Rectangular function:
    f(t) = 1/T for 0 <= t <= dt
           0   otherwise
    Works with scalar or NumPy array t.
    """
    t = np.asarray(t)
    offset = t[len(t)//2]
    return np.where((t >= offset-dt) & (t <= offset + dt), 1.0, 0.0)

def conv_circ( signal, ker ):
    return np.real( ifftshift( ifft( fft(signal) * fft(ker) )) )

# Parameters

In [ ]:
dt = 0.297619  # ns
nbin = 81
period = nbin * dt
t = np.arange(nbin)*dt
tau = 5
tau_R = 1.5
C = 1e3
C_R = 3e3
dT = 50*dt

In [ ]:
irf = rectangular_IRF(t, 2*dt)
irf /= irf.sum()  # Normalize the IRF to have an area of 1

In [ ]:
irf = gaussian(irf, 1)

In [ ]:
plt.plot(t, irf)

# Tests

In [ ]:
pure_model = model(t, C, tau, period)

In [ ]:
plt.plot(t, pure_model)

In [ ]:
corrected_ref = corrected_ref_model(t, C, C_R, tau, tau_R, period)

In [ ]:
plt.plot(t, corrected_ref, label='model ref')
plt.legend()

In [ ]:
ref_data = conv_circ(irf, model(t, C_R, tau_R, period)) #np.random.poisson(conv_circ(irf, pure_model))

In [ ]:
plt.plot(t, pure_model, label='model')
plt.plot(t, ref_data, label='ref')
plt.legend()

In [ ]:
ref_data_shifted = shift(ref_data, dT, order=1, mode='grid-wrap')
massimo = np.sum(ref_data_shifted)
data_with_ref = np.random.poisson(conv_circ(ref_data_shifted/massimo, corrected_ref))

In [ ]:
plt.plot(t, corrected_ref, label='ciclic model')
plt.plot(t, ref_data_shifted, label='ref dato shifted')
plt.plot(t, data_with_ref, label='convolved model with ref')
plt.legend()

# Fitting with same model

In [ ]:
def fit_model(t, C, C_R, dT, tau, ref_data=ref_data, tau_R=tau_R, period=period):
    model_ref = corrected_ref_model(t, C, C_R, tau, tau_R, period)
    model_ref_shifted = shift(model_ref, dT, order=1, mode='grid-wrap') #shift of corrected model
    model = conv_circ(model_ref_shifted, ref_data) 
    return model

In [ ]:
popt, conv = scipy.optimize.curve_fit(
    fit_model, t, data_with_ref, p0=[10, 400, 25, 10], bounds=([0, 0, -nbin//2, 0.01], [np.inf, np.inf, nbin//2, 10])
)
 
fitted_model_data_ref = fit_model(t, *popt)

In [ ]:
print(C, C_R, dT, tau)
print(popt)

In [ ]:
plt.plot(t, fitted_model_data_ref, label='fitted model')
plt.plot(t, data_with_ref, label='convolved model')
plt.legend()

# Fit dato generated with IRF

In [ ]:
def fit_model_ref(t, C, C_R, dT, tau, ref_data=ref_data, tau_R=tau_R, period=period):
    model = ref_model(t, C, C_R, tau, tau_R, dT, ref_data, period)
    return model

In [ ]:
model_dato_irf = shift(model(t, C, tau, period), dT, order=1, mode='grid-wrap')
data_with_irf = np.random.poisson(conv_circ(model_dato_irf, irf))

In [ ]:
plt.plot(t, data_with_irf, label='data with irf')
plt.plot(t, data_with_ref, label='data with ref')
plt.legend()

In [ ]:
popt_irf, conv = scipy.optimize.curve_fit(
    fit_model_ref, t, data_with_irf, p0=[10, 200, 25, 5], bounds=([0, 0, -nbin//2, 0.01], [np.inf, np.inf, nbin//2, 10])
)
 
fitted_model_data_irf = fit_model_ref(t, *popt_irf)

In [ ]:
print(C, C_R, dT, tau)
print(*popt_irf)

In [ ]:
fitted_model_data_irf = fit_model_ref(t, *popt_irf)

In [ ]:
fitted_model_giusto = fit_model_ref(t, C, C_R, dT, tau )

In [ ]:
plt.plot(t, fitted_model_data_irf, label='fitted model')
plt.plot(t, data_with_irf, label='data')
plt.legend()